In [14]:
!pip install -q langchain langchain-groq langchain-huggingface chromadb pandas

In [15]:
!pip install -q langchain-text-splitters
!pip install -q langchain_community

In [16]:
import os
import pandas as pd
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage
print("Utilities Downloaded")

Utilities Downloaded


In [17]:
os.environ["GROQ_API_KEY"] = "GROQ_API_KEY"
raw_url = "https://raw.githubusercontent.com/JaviRute/top_1000_movies-data_science_project/main/imdb_top_1000.csv"
df = pd.read_csv(raw_url).fillna("Not Available")
print(f"Dataset Loaded!")
print(f"   Total Movies : {len(df)}")
print(f"   Columns      : {df.columns.tolist()}")
print(f"\n=== Movies ===")
print(df[['Series_Title','Director','Star1','IMDB_Rating']].head(3))

Dataset Loaded!
   Total Movies : 1001
   Columns      : ['Poster_Link', 'Series_Title', 'Released_Year', 'Certificate', 'Runtime', 'Genre', 'IMDB_Rating', 'Overview', 'Meta_score', 'Director', 'Star1', 'Star2', 'Star3', 'Star4', 'No_of_Votes', 'Gross', 'Number_of_Movies']

=== Movies ===
               Series_Title              Director           Star1  IMDB_Rating
0  The Shawshank Redemption        Frank Darabont     Tim Robbins          9.3
1             The Godfather  Francis Ford Coppola   Marlon Brando          9.2
2           The Dark Knight     Christopher Nolan  Christian Bale          9.0


In [18]:
def create_movie_document(row):
    """
    Takes one CSV row and creates a rich text Document.
    Rich text = better RAG answers.
    """
    title    = row.get('Series_Title', 'Not Available')
    year     = row.get('Released_Year', 'Not Available')
    genre    = row.get('Genre', 'Not Available')
    rating   = row.get('IMDB_Rating', 'Not Available')
    director = row.get('Director', 'Not Available')
    star1    = row.get('Star1', '')
    star2    = row.get('Star2', '')
    star3    = row.get('Star3', '')
    star4    = row.get('Star4', '')
    plot     = row.get('Overview', 'Not Available')
    runtime  = row.get('Runtime', 'Not Available')
    actors   = ", ".join(filter(None, [star1, star2, star3, star4]))
    content = f"""
    Movie Title: {title}
    Released Year: {year}
    Genre: {genre}
    IMDB Rating: {rating}/10
    Runtime: {runtime}
    Director: {director}
    Lead Actors: {actors}
    Plot Summary: {plot}
    """
    metadata = {
        "title"   : str(title),
        "year"    : str(year),
        "genre"   : str(genre),
        "rating"  : str(rating),
        "director": str(director)
    }

    return Document(page_content=content, metadata=metadata)
documents = [create_movie_document(row) for _, row in df.iterrows()]
print(f" Created {len(documents)} documents from CSV")

 Created 1001 documents from CSV


In [19]:
custom_movies = [
    Document(
        page_content="""
        Movie Title: Baahubali: The Beginning
        Released Year: 2015
        Genre: Action, Drama, Fantasy
        IMDB Rating: 8.0/10
        Runtime: 159 min
        Director: S. S. Rajamouli
        Lead Actors: Prabhas, Rana Daggubati, Anushka Shetty, Tamannaah
        Plot Summary: In ancient India, an adventurous and daring man
        becomes involved in a legend of the land and finds his true
        heritage as a mighty warrior prince named Baahubali.
        He fights against the cruel king Bhallaladeva played by Rana Daggubati
        to reclaim his rightful throne.
        Devasena played by Anushka Shetty is the princess he loves.
        """,
        metadata={
            "title"   : "Baahubali: The Beginning",
            "year"    : "2015",
            "genre"   : "Action, Drama, Fantasy",
            "rating"  : "8.0",
            "director": "S. S. Rajamouli"
        }
    ),
    Document(
        page_content="""
        Movie Title: Chhichhore
        Released Year: 2019
        Genre: Comedy, Drama
        IMDB Rating: 8.3/10
        Runtime: 143 min
        Director: Nitesh Tiwari
        Lead Actors: Sushant Singh Rajput, Shraddha Kapoor,
        Varun Sharma, Prateik Babbar
        Plot Summary: A man narrates his college days to his son
        who is recovering in hospital after failing his exam.
        The hero Anni played by Sushant Singh Rajput recalls
        his hostel life and friendship with a group of losers
        who become winners. A heartwarming story about
        not giving up in life and friendship.
        """,
        metadata={
            "title"   : "Chhichhore",
            "year"    : "2019",
            "genre"   : "Comedy, Drama",
            "rating"  : "8.3",
            "director": "Nitesh Tiwari"
        }
    ),

    Document(
        page_content="""
        Movie Title: Saaho
        Released Year: 2019
        Genre: Action, Thriller
        IMDB Rating: 5.3/10
        Runtime: 170 min
        Director: Sujeeth
        Lead Actors: Prabhas, Shraddha Kapoor,
        Jackie Shroff, Neil Nitin Mukesh
        Plot Summary: An undercover cop named Ashok played by Prabhas
        infiltrates a crime syndicate to recover stolen money
        and bring down the criminal network.
        Shraddha Kapoor plays the female lead named Amritha.
        A high octane action thriller with stunning visuals.
        """,
        metadata={
            "title"   : "Saaho",
            "year"    : "2019",
            "genre"   : "Action, Thriller",
            "rating"  : "5.3",
            "director": "Sujeeth"
        }
    )
]
documents.extend(custom_movies)
print(f" Custom Movies Added!")
print(f"   IMDB Movies   : {len(df)}")
print(f"   Custom Movies : {len(custom_movies)}")
print(f"   Total         : {len(documents)}")

 Custom Movies Added!
   IMDB Movies   : 1001
   Custom Movies : 3
   Total         : 1004


In [20]:
print("Chunking documents...")
print("(This takes about 30 seconds...)\n")
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=60,
    separators=["\n", ". ", " "]
)
chunks = splitter.split_documents(documents)
print(f" Chunking Complete")
print(f"   Total Documents : {len(documents)}")
print(f"   Total Chunks    : {len(chunks)}")
print(f"   Avg Chunks/Doc  : {len(chunks)/len(documents):.1f}")
print(f"\n=== Chhihhore chunks ===")
chichore_chunks = [c for c in chunks if "Chhichhore" in c.page_content]
print(f"Chhichhore chunks found: {len(chichore_chunks)}")
for i, chunk in enumerate(chichore_chunks):
    print(f"\nChunk {i+1}:")
    print(chunk.page_content[:150])

Chunking documents...
(This takes about 30 seconds...)

 Chunking Complete
   Total Documents : 1004
   Total Chunks    : 1491
   Avg Chunks/Doc  : 1.5

=== Chhihhore chunks ===
Chhichhore chunks found: 2

Chunk 1:
Movie Title: Chhichhore
    Released Year: 2019
    Genre: Comedy, Drama
    IMDB Rating: 8.2/10
    Runtime: 143 min
    Director: Nitesh Tiwari
    

Chunk 2:
Movie Title: Chhichhore
        Released Year: 2019
        Genre: Comedy, Drama
        IMDB Rating: 8.3/10
        Runtime: 143 min
        Director


In [21]:
print("Loading embedding model...")
print("(First time: downloads ~90MB, takes 2 mins)")
print("(After that: loads from cache instantly)\n")
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)
print(" Embedding model loaded")
print("\nBuilding ChromaDB vector database...")
print("(This takes 2-3 minutes for 1000+ movies...)\n")
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)
retriever = vector_db.as_retriever(
    search_kwargs={"k": 3}
)
print(" ChromaDB ready!")
print(f"   Stored {len(chunks)} chunks in database")
print(f"   Retriever will return top 3 matches per query")

Loading embedding model...
(First time: downloads ~90MB, takes 2 mins)
(After that: loads from cache instantly)



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

 Embedding model loaded

Building ChromaDB vector database...
(This takes 2-3 minutes for 1000+ movies...)

 ChromaDB ready!
   Stored 1491 chunks in database
   Retriever will return top 3 matches per query


In [22]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)
print(" LLM (Llama 3) ready!")
memory_prompt = ChatPromptTemplate.from_messages([
    ("system", """
    You are an expert Movie Recommender and Information System.
    You have a database of top 1000 IMDB movies plus Indian movies
    like Baahubali, Chhichhore and Saaho.

    RULES:
    1. Answer ONLY from the context provided below
    2. Use chat history to understand follow up questions
    3. When a question is given to you review the database with question and if the the answer to the question is not available in the database say your question is wrong
    4. If movie not in database say: "This movie is not in my database"
    5. Be conversational and helpful
    6. When user says "this movie" or "that film" use chat history to identify which movie
    7. Do not use prior knowledge do not infer missing facts
    8. Do not speculate
    9. If there no backed evidence from the database simply say :"i dont know",but do not try to map semantically with the appropriate data
    10. If there the movie in the database but dont know about the query return movie is in database but dont know the query
    Context from movie database:
    {context}
    """),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])
print(" Memory prompt created!")

 LLM (Llama 3) ready!
 Memory prompt created!


In [23]:
from operator import itemgetter

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

memory_rag_chain = (
    {
        "context": itemgetter("question") | retriever | format_docs,
        "question": itemgetter("question"),
        "chat_history": itemgetter("chat_history")
    }
    | memory_prompt
    | llm
    | StrOutputParser()
)

In [24]:
class MovieChatbot:
    """
    A Movie RAG Chatbot with conversation memory.

    How memory works:
    → Starts empty []
    → After each exchange adds:
       HumanMessage(your question)
       AIMessage(bot answer)
    → Sends full history with every new question
    → AI always has full conversation context
    → Oldest messages removed after max_history
    """
    def __init__(self, rag_chain, max_history=10):
        self.rag_chain    = rag_chain
        self.chat_history = []
        self.max_history  = max_history
        self.question_count = 0
        print("MovieChatbot initialized")
        print(f"Memory: Stores last {max_history} exchanges")
        print(f"Each exchange = 1 question + 1 answer\n")
    def chat(self, question, show_thinking=False):
        """
        Ask a question. Memory handled automatically.

        Parameters:
            question     : Your movie question (string)
            show_thinking: Shows memory state if True
        """
        self.question_count += 1
        if show_thinking:
            print(f"\n{'─'*50}")
            print(f"Memory before Question {self.question_count}:")
            print(f"Messages stored: {len(self.chat_history)}")

            if self.chat_history:
                print("   Last exchange:")
                last_human = None
                last_ai    = None
                for msg in self.chat_history:
                    if isinstance(msg, HumanMessage):
                        last_human = msg.content
                    else:
                        last_ai = msg.content
                if last_human:
                    print(f"{last_human[:50]}...")
                if last_ai:
                    print(f"{last_ai[:50]}...")
            else:
                print("   (Empty - this is your first question!)")
            print(f"{'─'*50}")
        response = self.rag_chain.invoke({
            "question"    : question,
            "chat_history": self.chat_history
        })
        self.chat_history.append(HumanMessage(content=question))
        self.chat_history.append(AIMessage(content=response))
        if len(self.chat_history) > self.max_history * 2:
            removed = self.chat_history[:2]
            self.chat_history = self.chat_history[2:]
            print(f"[Memory: Removed oldest exchange to save space]")
        return response
    def clear_memory(self):
        """Start a completely fresh conversation"""
        self.chat_history   = []
        self.question_count = 0
        print("Memory cleared! Fresh conversation started.\n")
    def show_memory(self):
        """Display everything currently in memory"""
        if not self.chat_history:
            print("Memory is empty!")
            return
        print(f"\n{'='*50}")
        print(f"CURRENT MEMORY ({len(self.chat_history)//2} exchanges)")
        print(f"{'='*50}")
        for i, msg in enumerate(self.chat_history):
            if isinstance(msg, HumanMessage):
                print(f"\n👤 You    : {msg.content[:100]}")
            else:
                print(f"🤖 Bot    : {msg.content[:100]}...")
    def debug_retrieval(self, question):
        """
        Shows exactly what chunks are retrieved for a question.
        Use this when answers seem wrong.
        Professional debugging tool.
        """
        print(f"\n DEBUG: What does retriever find for:")
        print(f"   '{question}'")
        print(f"{'─'*50}")
        retrieved = retriever.invoke(question)
        for i, chunk in enumerate(retrieved):
            title = chunk.metadata.get('title', 'Unknown')
            print(f"\nChunk {i+1}: [{title}]")
            print(chunk.page_content[:200])
            print("...")
movie_bot = MovieChatbot(memory_rag_chain, max_history=10)

MovieChatbot initialized
Memory: Stores last 10 exchanges
Each exchange = 1 question + 1 answer



In [25]:
questions_test1 = [
    "why did kattapa killed bahubali in mr perfect",
    "about movie saaho"]
for q in questions_test1:
    print(f"\nYou: {q}")
    answer = movie_bot.chat(q)
    print(f"Bot: {answer}")


You: why did kattapa killed bahubali in mr perfect
Bot: I don't know.

You: about movie saaho
Bot: Saaho is in my database. It's an action thriller film released in 2019, starring Prabhas and Shraddha Kapoor. The movie follows the story of an undercover cop named Ashok who infiltrates a crime syndicate to recover stolen money. Would you like to know more about the movie, such as its director, runtime, or IMDB rating?


In [26]:
print("\nTEST 2: MEMORY VISUALIZATION")
print("Clearing memory for clean test...")
movie_bot.clear_memory()
questions_test2 = [
    "why did bhallaladeva removed bahubali from the postion army cheif",
]
for q in questions_test2:
    print(f"\nYou: {q}")
    answer = movie_bot.chat(q, show_thinking=True)
    print(f"Bot: {answer}")


TEST 2: MEMORY VISUALIZATION
Clearing memory for clean test...
Memory cleared! Fresh conversation started.


You: why did bhallaladeva removed bahubali from the postion army cheif

──────────────────────────────────────────────────
Memory before Question 1:
Messages stored: 0
   (Empty - this is your first question!)
──────────────────────────────────────────────────
Bot: I don't know.
